In [5]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # save figures without needing a display
import matplotlib.pyplot as plt

In [6]:
# ----------------------------- Settings -----------------------------
FILE_PATH = "/mnt/user-data/uploads/School_Data_CSV.csv"  # change to a local path if needed
LAT_RANGE = (-35.0, -22.0)   # plausible South African latitude (degrees)
LON_RANGE = (16.0, 33.0)     # plausible South African longitude (degrees)
TOP_N_WARDS = 30             # wards shown in the line plot (see plot_quintiles_across_wards)
QUINTILES = [1, 2, 3, 4, 5]
WARD_PLACEHOLDERS = {"", "TO BE UPDATED", "99", "99.0", "UNKNOWN", "NOT APPLICABLE",
                     "N/A", "NA", "NAN", "NULL", "NONE", "."}

# UNIT OF COUNTING: one "school" = one unique emisno inside the group being counted
# (a quintile, or a ward-quintile pair). Rows are NOT schools: emisno has ~33,670 unique
# values across ~200k rows, so counting rows would massively inflate the numbers.


def log_step(step, before, after):
    """Print how many records a step excluded."""
    print(f"{step:<58} excluded: {before - after:>8,}  | remaining: {after:>8,}")

In [7]:
# ----------------------------- Data preparation -----------------------------
def load_data(path):
    """Load everything as text so that no value is silently coerced on read."""
    df = pd.read_csv(path, dtype=str)
    print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
    return df


def remove_duplicates(df):
    """Drop exact duplicate rows, then trim whitespace in text columns."""
    before = len(df)
    df = df.drop_duplicates().copy()
    log_step("Exact duplicate rows", before, len(df))
    for col in df.columns:
        df[col] = df[col].str.strip()
    return df


def drop_missing_emisno(df):
    """emisno is the school identifier, so rows without it cannot be counted."""
    before = len(df)
    df = df[df["emisno"].notna() & (df["emisno"] != "")].copy()
    log_step("Missing emisno", before, len(df))
    return df


def clean_quintile(df):
    """Map 1-5, Q1-Q5, NQ1-NQ5 to integers 1-5; everything else is excluded."""
    before = len(df)
    text = df["quintile"].astype("string").str.upper()
    extracted = text.str.extract(r"^(?:NQ|Q)?\s*([1-5])(?:\.0+)?$")[0]
    df = df.assign(quintile=pd.to_numeric(extracted, errors="coerce"))
    df = df.dropna(subset=["quintile"]).copy()
    df["quintile"] = df["quintile"].astype(int)
    log_step("Invalid/placeholder quintile ('TO BE UPDATED', '99', ...)", before, len(df))
    return df


def clean_ward(df):
    """Exclude placeholder ward_id values (only needed for the ward plot)."""
    before = len(df)
    upper = df["ward_id"].astype("string").str.upper().fillna("")
    df = df[~upper.isin(WARD_PLACEHOLDERS)].copy()
    log_step("Placeholder/missing ward_id", before, len(df))
    return df


def clean_coordinates(df):
    """Convert coordinates to numeric and keep only plausible South African values."""
    df = df.copy()
    df["new_lat"] = pd.to_numeric(df["new_lat"], errors="coerce")
    df["new_long"] = pd.to_numeric(df["new_long"], errors="coerce")

    before = len(df)
    df = df.dropna(subset=["new_lat", "new_long"]).copy()
    log_step("Missing / non-numeric coordinates", before, len(df))

    before = len(df)
    in_range = (df["new_lat"].between(*LAT_RANGE)) & (df["new_long"].between(*LON_RANGE))
    df = df[in_range].copy()
    log_step(f"Outside lat {LAT_RANGE} / long {LON_RANGE}", before, len(df))

    before = len(df)
    df = df.drop_duplicates(subset=["emisno", "new_lat", "new_long"])
    log_step("Repeated emisno at identical coordinates", before, len(df))
    return df

In [8]:
# ----------------------------- (a) Line plot -----------------------------
def plot_quintiles_across_wards(df_ward, out_file="plot_a_quintiles_by_ward.png"):
    """
    Line plot: schools per quintile across wards.
    LIMITATIONS: wards are nominal (no natural order), so a line joining them implies a
    trend that does not exist; the lines only help compare quintile profiles.
    ~4,800 wards cannot be read on one axis, so I show the TOP_N_WARDS wards with the
    most unique schools, ordered from largest to smallest, so the order is meaningful
    (ranked by size). The remaining wards are not shown.
    """
    ward_totals = df_ward.groupby("ward_id")["emisno"].nunique().nlargest(TOP_N_WARDS)
    subset = df_ward[df_ward["ward_id"].isin(ward_totals.index)]

    counts = (subset.groupby(["ward_id", "quintile"])["emisno"].nunique()
                    .unstack("quintile")
                    .reindex(index=ward_totals.index, columns=QUINTILES)
                    .fillna(0))

    fig, ax = plt.subplots(figsize=(13, 6))
    for quintile in QUINTILES:
        ax.plot(counts.index.astype(str), counts[quintile], marker="o", label=f"Quintile {quintile}")
    ax.set_title(f"Unique schools per quintile across the {TOP_N_WARDS} largest wards")
    ax.set_xlabel("Ward ID (ordered by total unique schools, largest first)")
    ax.set_ylabel("Number of unique schools (distinct emisno)")
    ax.tick_params(axis="x", rotation=90)
    ax.legend(title="Quintile")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_file, dpi=150)
    plt.close(fig)
    print(f"Saved {out_file}")

In [9]:
# ----------------------------- (b) Scatter plot -----------------------------
def plot_school_locations(df_geo, out_file="plot_b_school_locations.png"):
    """Scatter of school locations (longitude on x, latitude on y), coloured by quintile."""
    fig, ax = plt.subplots(figsize=(9, 8))
    colours = plt.cm.tab10.colors
    for index, quintile in enumerate(QUINTILES):
        part = df_geo[df_geo["quintile"] == quintile]
        ax.scatter(part["new_long"], part["new_lat"], s=3, alpha=0.5,
                   color=colours[index], label=f"Quintile {quintile} (n={len(part):,})")
    ax.set_title("Geographic locations of schools in South Africa")
    ax.set_xlabel("Longitude (new_long, degrees)")
    ax.set_ylabel("Latitude (new_lat, degrees)")
    ax.set_aspect("equal", adjustable="datalim")
    ax.legend(title="Quintile", markerscale=4, loc="lower left")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_file, dpi=150)
    plt.close(fig)
    print(f"Saved {out_file}")

In [10]:
# ----------------------------- (c) Histogram -----------------------------
def plot_quintile_histogram(df_quintile, out_file="plot_c_quintile_histogram.png"):
    """
    Quintile is a categorical (ordinal) variable, so a classic histogram of a continuous
    variable is not strictly appropriate. I treat it as a discrete variable: one bin per
    quintile (edges at 0.5, 1.5, ...), centred on the integer labels, with gaps between
    bars. This is equivalent to a bar chart of counts per category.
    """
    schools = df_quintile.drop_duplicates(subset=["emisno", "quintile"])
    bin_edges = np.arange(0.5, 6.5, 1)

    fig, ax = plt.subplots(figsize=(8, 5))
    counts, _, bars = ax.hist(schools["quintile"], bins=bin_edges, rwidth=0.8,
                              color="steelblue", edgecolor="black")
    ax.bar_label(bars, labels=[f"{int(c):,}" for c in counts])
    ax.set_xticks(QUINTILES)
    ax.set_title("Distribution of schools by quintile")
    ax.set_xlabel("Quintile (1 = poorest, 5 = least poor)")
    ax.set_ylabel("Number of unique schools (distinct emisno)")
    fig.tight_layout()
    fig.savefig(out_file, dpi=150)
    plt.close(fig)
    print(f"Saved {out_file}")


In [13]:
# ----------------------------- Main -----------------------------
def main():
    raw = load_data(FILE_PATH)
    deduped = remove_duplicates(raw)
    with_id = drop_missing_emisno(deduped)

    # Shared base: valid quintile. Used by (c) directly and by (a)/(b) below.
    df_quintile = clean_quintile(with_id)

    # (a) additionally needs a valid ward; (b) additionally needs valid coordinates.
    # Done as separate branches so (c) is not shrunk by filters it does not need.
    print("\n-- Extra cleaning for plot (a): ward --")
    df_ward = clean_ward(df_quintile)

    print("\n-- Extra cleaning for plot (b): coordinates --")
    df_geo = clean_coordinates(df_quintile)

    # Diagnostic: emisno values with conflicting valid quintiles
    conflicts = (df_quintile.groupby("emisno")["quintile"].nunique() > 1).sum()
    print(f"\nUnique emisno (valid quintile): {df_quintile['emisno'].nunique():,}")
    print(f"emisno with more than one valid quintile: {conflicts:,}\n")

    plot_quintiles_across_wards(df_ward)
    plot_school_locations(df_geo)
    plot_quintile_histogram(df_quintile)


